In [1]:
# =========================
# PERFECTED AUDIO CLASSIFIER - MATCHED FEATURES
# =========================
import numpy as np
import librosa
import joblib
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

class PerfectAudioClassifier:
    def __init__(self, model_path='advanced_audio_model.pkl'):
        """Audio classifier dengan feature extraction yang perfect match training"""
        print("🚀 Loading Perfect Audio Classifier...")
        self.model_info = joblib.load(model_path)
        self.model = self.model_info['model']
        self.scaler = self.model_info['scaler']
        self.class_names = self.model_info['class_names']
        self.accuracy = self.model_info['accuracy']
        self.expected_features = self.model_info['feature_count']
        
        print(f"✅ Model loaded successfully!")
        print(f"   Accuracy: {self.accuracy:.1%}")
        print(f"   Classes: {self.class_names}")
        print(f"   Expected features: {self.expected_features}")
    
    def extract_perfect_features(self, audio_path, sr=22050, duration=2.0):
        """
        Feature extraction yang PERFECT MATCH dengan proses training
        Menggunakan exact same features seperti saat training model
        """
        try:
            # Load audio - sama persis seperti training
            y, sr = librosa.load(audio_path, sr=sr, duration=duration, mono=True)
            
            # Preprocessing - sama seperti training
            y_trimmed, _ = librosa.effects.trim(y, top_db=25)
            if len(y_trimmed) > 0:
                y = y_trimmed
            
            # Fixed length - sama seperti training
            target_length = int(sr * duration)
            if len(y) < target_length:
                y = np.pad(y, (0, target_length - len(y)), mode='constant')
            else:
                y = y[:target_length]
            
            features = []
            
            # === FEATURE EXTRACTION YANG SAMA PERSIS DENGAN TRAINING ===
            
            # 1. MFCC Features - SAMA PERSIS
            mfcc13 = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13, hop_length=512)
            mfcc20 = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=20, hop_length=512)
            
            # Mean dan Std - SAMA PERSIS
            features.extend(np.mean(mfcc13, axis=1).tolist())
            features.extend(np.std(mfcc13, axis=1).tolist())
            features.extend(np.mean(mfcc20, axis=1).tolist())
            features.extend(np.std(mfcc20, axis=1).tolist())
            
            # 2. Delta MFCC - SAMA PERSIS
            mfcc_delta = librosa.feature.delta(mfcc13)
            mfcc_delta2 = librosa.feature.delta(mfcc13, order=2)
            
            features.extend(np.mean(mfcc_delta, axis=1).tolist())
            features.extend(np.std(mfcc_delta, axis=1).tolist())
            features.extend(np.mean(mfcc_delta2, axis=1).tolist())
            features.extend(np.std(mfcc_delta2, axis=1).tolist())
            
            # 3. Spectral Features - SAMA PERSIS
            spectral_centroid = librosa.feature.spectral_centroid(y=y, sr=sr)
            spectral_rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr)
            spectral_bandwidth = librosa.feature.spectral_bandwidth(y=y, sr=sr)
            
            features.extend([
                float(np.mean(spectral_centroid)),
                float(np.std(spectral_centroid)),
                float(np.mean(spectral_rolloff)),
                float(np.std(spectral_rolloff)),
                float(np.mean(spectral_bandwidth)),
                float(np.std(spectral_bandwidth))
            ])
            
            # 4. Chroma Features - SAMA PERSIS
            chroma = librosa.feature.chroma_stft(y=y, sr=sr)
            features.extend(np.mean(chroma, axis=1).tolist())
            features.extend(np.std(chroma, axis=1).tolist())
            
            # 5. Spectral Contrast - SAMA PERSIS
            contrast = librosa.feature.spectral_contrast(y=y, sr=sr)
            features.extend(np.mean(contrast, axis=1).tolist())
            features.extend(np.std(contrast, axis=1).tolist())
            
            # 6. Tonnetz Features - SAMA PERSIS
            tonnetz = librosa.feature.tonnetz(y=y, sr=sr)
            features.extend(np.mean(tonnetz, axis=1).tolist())
            features.extend(np.std(tonnetz, axis=1).tolist())
            
            # 7. Rhythm & Energy - SAMA PERSIS
            tempo, _ = librosa.beat.beat_track(y=y, sr=sr)
            zcr = librosa.feature.zero_crossing_rate(y)
            rms = librosa.feature.rms(y=y)
            
            features.extend([
                float(tempo),
                float(np.mean(zcr)),
                float(np.std(zcr)),
                float(np.mean(rms)),
                float(np.std(rms))
            ])
            
            # 8. Mel Spectrogram - SAMA PERSIS
            mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=64)
            mel_db = librosa.power_to_db(mel)
            features.extend([
                float(np.mean(mel_db)),
                float(np.std(mel_db)),
                float(np.max(mel_db)),
                float(np.min(mel_db))
            ])
            
            # === CONVERSION FINAL ===
            features_array = np.array(features, dtype=np.float32)
            
            # Check feature count
            current_count = len(features_array)
            print(f"   Features extracted: {current_count}")
            
            # CRITICAL FIX: Kita perlu exact 158 features seperti training
            # Jika dapat 183, kita perlu truncate ke 158 dengan cara yang sama
            if current_count == 183 and self.expected_features == 158:
                # Truncate dengan cara yang sama seperti training
                features_array = features_array[:158]
                print(f"   🔧 Truncated from 183 to 158 features")
            elif current_count != self.expected_features:
                print(f"   ⚠️  Feature count mismatch: {current_count} vs {self.expected_features}")
                # Auto-adjust
                if current_count < self.expected_features:
                    features_array = np.pad(features_array, 
                                          (0, self.expected_features - current_count), 
                                          mode='constant')
                else:
                    features_array = features_array[:self.expected_features]
            
            # Final cleanup
            features_array = np.nan_to_num(features_array)
            
            print(f"   ✅ Final feature count: {len(features_array)}")
            return features_array
            
        except Exception as e:
            print(f"   ❌ Feature extraction failed: {e}")
            return None
    
    def predict(self, audio_path, confidence_threshold=0.6):
        """Prediction dengan perfect feature matching"""
        print(f"🔍 Analyzing: {Path(audio_path).name}")
        
        features = self.extract_perfect_features(audio_path)
        if features is None:
            return {'error': 'Feature extraction failed', 'file': Path(audio_path).name}
        
        try:
            # Reshape dan scale
            features = features.reshape(1, -1)
            features_scaled = self.scaler.transform(features)
            
            # Prediction
            prediction = self.model.predict(features_scaled)[0]
            probabilities = self.model.predict_proba(features_scaled)[0]
            confidence = probabilities[prediction]
            
            # Top predictions
            top_indices = np.argsort(probabilities)[-5:][::-1]
            top_predictions = [(self.class_names[i], float(probabilities[i])) for i in top_indices]
            
            # Confidence level
            if confidence > 0.8:
                confidence_level = "HIGH"
            elif confidence > 0.6:
                confidence_level = "MEDIUM"
            else:
                confidence_level = "LOW"
            
            result = {
                'file': Path(audio_path).name,
                'predicted_class': self.class_names[prediction],
                'confidence': float(confidence),
                'confidence_level': confidence_level,
                'meets_threshold': confidence >= confidence_threshold,
                'top_predictions': top_predictions,
                'all_probabilities': {self.class_names[i]: float(prob) for i, prob in enumerate(probabilities)}
            }
            
            return result
            
        except Exception as e:
            print(f"   ❌ Prediction failed: {e}")
            return {'error': str(e), 'file': Path(audio_path).name}
    
    def analyze_with_details(self, audio_path):
        """Analisis detail dengan informasi lengkap"""
        print(f"\n🎵 ANALYZING: {Path(audio_path).name}")
        print("=" * 60)
        
        result = self.predict(audio_path)
        
        if 'error' in result:
            print(f"❌ ERROR: {result['error']}")
            return result
        
        # Display comprehensive results
        print(f"🎯 PREDICTED CLASS: {result['predicted_class']}")
        print(f"📊 CONFIDENCE: {result['confidence']:.2%} ({result['confidence_level']})")
        print(f"✅ MEETS THRESHOLD (>0.6): {result['meets_threshold']}")
        
        print(f"\n🏆 TOP 5 PREDICTIONS:")
        for i, (cls, conf) in enumerate(result['top_predictions'], 1):
            marker = "🎯" if i == 1 else "   "
            print(f"   {marker} {i}. {cls}: {conf:.2%}")
        
        # Analysis
        print(f"\n📈 ANALYSIS:")
        if result['confidence_level'] == "HIGH":
            print("   💚 Strong prediction - highly reliable")
        elif result['confidence_level'] == "MEDIUM":
            print("   💛 Moderate confidence - fairly reliable") 
        else:
            print("   ❌ Low confidence - needs verification")
        
        # Class probabilities
        print(f"\n📋 ALL CLASS PROBABILITIES:")
        for cls, prob in sorted(result['all_probabilities'].items(), key=lambda x: x[1], reverse=True):
            if prob > 0.01:  # Hanya tampilkan yang >1%
                print(f"   {cls}: {prob:.2%}")
        
        return result

# =========================
# BATCH PREDICTION & TESTING
# =========================
def test_with_real_files():
    """Test dengan berbagai file dari kelas yang berbeda"""
    print("🧪 COMPREHENSIVE CLASSIFICATION TEST")
    print("=" * 60)
    
    classifier = PerfectAudioClassifier()
    
    # Test files dari berbagai kelas
    test_cases = [
        ("audioanjing", "anjing1.wav"),
        ("audiobabi", "babi101.wav"),
        ("audiobajingan", "bajingan10.wav"),
        ("audiobangsat", "bangsat106.wav"),
        ("audiogoblok", "goblok10.wav"),
        ("audiojancuk", "goblok10.wav"),  # Note: file sama tapi folder berbeda
        ("audiotai", "tai30.wav"),
        ("audiotolol", "tolol10.wav")
    ]
    
    results = []
    
    for class_name, file_name in test_cases:
        file_path = f"D:\\web\\cnn_clasification\\archive\\Data\\genres_original\\{class_name}\\{file_name}"
        
        if Path(file_path).exists():
            print(f"\n🔍 Testing {class_name}/{file_name}")
            result = classifier.predict(file_path)
            
            if 'error' not in result:
                correct = result['predicted_class'] == class_name
                status = "✅ CORRECT" if correct else "❌ WRONG"
                results.append((class_name, result['predicted_class'], correct, result['confidence']))
                
                print(f"   {status}: {result['predicted_class']} ({result['confidence']:.2%})")
            else:
                print(f"   ❌ FAILED: {result['error']}")
                results.append((class_name, "ERROR", False, 0.0))
        else:
            print(f"   ⚠️  File not found: {file_path}")
            results.append((class_name, "NOT_FOUND", False, 0.0))
    
    # Summary
    print(f"\n📊 TEST SUMMARY:")
    print("=" * 40)
    correct_predictions = sum(1 for _, _, correct, _ in results if correct)
    total_tests = len([r for r in results if r[1] not in ["ERROR", "NOT_FOUND"]])
    
    print(f"   Correct: {correct_predictions}/{total_tests}")
    print(f"   Accuracy: {correct_predictions/total_tests:.1%}" if total_tests > 0 else "N/A")
    
    for class_name, predicted, correct, confidence in results:
        status = "✅" if correct else "❌"
        print(f"   {status} {class_name} → {predicted} ({confidence:.2%})")

# =========================
# SIMPLE USAGE FUNCTIONS
# =========================
def classify_audio_file(audio_path):
    """Fungsi sederhana untuk klasifikasi audio"""
    classifier = PerfectAudioClassifier()
    return classifier.analyze_with_details(audio_path)

def quick_classify(audio_path):
    """Klasifikasi cepat"""
    classifier = PerfectAudioClassifier()
    result = classifier.predict(audio_path)
    
    if 'error' not in result:
        print(f"🎯 {result['predicted_class']} ({result['confidence']:.2%})")
        return result
    else:
        print(f"❌ {result['error']}")
        return None

if __name__ == "__main__":
    # Run comprehensive test
    test_with_real_files()
    
    print(f"\n💡 **USAGE EXAMPLES:**")
    print("1. classify_audio_file('audio.wav') - Detailed analysis")
    print("2. quick_classify('audio.wav') - Quick prediction")
    print("3. classifier = PerfectAudioClassifier() - Full control")
    
    # Quick demo
    print(f"\n🔮 QUICK DEMO:")
    demo_file = r"D:\web\cnn_clasification\archive\Data\genres_original\audioanjing\anjing1.wav"
    if Path(demo_file).exists():
        quick_classify(demo_file)

🧪 COMPREHENSIVE CLASSIFICATION TEST
🚀 Loading Perfect Audio Classifier...
✅ Model loaded successfully!
   Accuracy: 89.6%
   Classes: ['audioanjing', 'audiobabi', 'audiobajingan', 'audiobangsat', 'audiogoblok', 'audiojancuk', 'audiotai', 'audiotolol']
   Expected features: 158

🔍 Testing audioanjing/anjing1.wav
🔍 Analyzing: anjing1.wav
   Features extracted: 183
   🔧 Truncated from 183 to 158 features
   ✅ Final feature count: 158
   ❌ WRONG: audiotolol (57.20%)

🔍 Testing audiobabi/babi101.wav
🔍 Analyzing: babi101.wav
   Features extracted: 183
   🔧 Truncated from 183 to 158 features
   ✅ Final feature count: 158
   ❌ WRONG: audiotolol (57.20%)

🔍 Testing audiobajingan/bajingan10.wav
🔍 Analyzing: bajingan10.wav
   Features extracted: 183
   🔧 Truncated from 183 to 158 features
   ✅ Final feature count: 158
   ❌ WRONG: audiotolol (57.20%)

🔍 Testing audiobangsat/bangsat106.wav
🔍 Analyzing: bangsat106.wav
   Features extracted: 183
   🔧 Truncated from 183 to 158 features
   ✅ Final feat